In [2]:
# fix for HF hub download
# see PR https://github.com/albumentations-team/albumentations/pull/2171
!pip install -U git+https://github.com/qubvel/albumentations@patch-2

  Cloning https://github.com/qubvel/albumentations (to revision patch-2) to /private/var/folders/2n/1f9qlz3d32536tpc82t1ty8c0000gn/T/pip-req-build-v8wrie5r
  Running command git clone --filter=blob:none --quiet https://github.com/qubvel/albumentations /private/var/folders/2n/1f9qlz3d32536tpc82t1ty8c0000gn/T/pip-req-build-v8wrie5r
  Running command git checkout -b patch-2 --track origin/patch-2
  Switched to a new branch 'patch-2'
  branch 'patch-2' set up to track 'origin/patch-2'.
  Resolved https://github.com/qubvel/albumentations to commit 6e40b35b92f6be8d9e5ccda5312fa9dcb0383a8c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
!pip install certifi
import certifi
import os
os.environ["SSL_CERT_FILE"] = certifi.where()

In [4]:
!pip install segmentation_models_pytorch

In [5]:
import requests
import numpy as np
import albumentations as A
import matplotlib.pyplot as plt
import torch
import segmentation_models_pytorch as smp

from PIL import Image

/Users/leeelyehezkel/PycharmProjects/SegFormer/.venv/lib/python3.9/site-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.6 (you have 1.4.21). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  Fil

In [6]:
import os

# Replace with your actual Hugging Face token
huggingface_token = "YOUR_HF_TOKEN"

# Set the token as an environment variable
os.environ["HF_TOKEN"] = huggingface_token

# You can verify that the token is set correctly
#print(os.environ.get("HF_TOKEN"))

# Restart the runtime.  This is essential for the change to take effect.
# In Colab, go to Runtime -> Restart runtime.
# You can also use a shell command to restart in some environments,
# but that's not reliable in all notebook environments.
# The manual restart is the most robust approach.


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#check for CUDA-enabled GPU
# More checkpoints can be found here:
# https://huggingface.co/collections/smp-hub/segformer-6749eb4923dea2c355f29a1f
checkpoint = "smp-hub/segformer-b2-1024x1024-city-160k" # hold identifier of pre-trained seg model hosted on Hugging Face

# Load pretrained model and preprocessing function
model = smp.from_pretrained(checkpoint).eval().to(device) #eval mode disables dropout, batch norm updates, etc
# dropout - disables some neurons to prevent overfitting
# batch - use fixed mean/variance instead of constantly updating them as in training

preprocessing = A.Compose.from_pretrained(checkpoint)
#loads corresponding image preprocessing pipeline from same checkpoint
#ensures that the preprocessing pipeline used during training is the same as the one used during inference

# Load image
url = "https://huggingface.co/datasets/hf-internal-testing/fixtures_ade20k/resolve/main/ADE_val_00000001.jpg"
image = Image.open(requests.get(url, stream=True).raw)
# fetches and opens image from URL

# Preprocess image
image = np.array(image)  # Converts to NumPy array
normalized_image = preprocessing(image=image)["image"]  # Applies preprocessing pipeline to image
input_tensor = torch.as_tensor(normalized_image, dtype=torch.float32)  # Converts to PyTorch tensor with explicit dtype
input_tensor = input_tensor.permute(2, 0, 1).unsqueeze(0)  # Converts dimension order from HWC -> BCHW
input_tensor = input_tensor.to(device)  # Moves input tensor to selected device (CPU or GPU)

# Perform inference
with torch.no_grad(): #disables grad calculations (not needed in inference because they are used to check how much each parameter contributed to error during trainin)
    output_mask = model(input_tensor) #runs model on input tensor

# Postprocess mask
mask = torch.nn.functional.interpolate(
    output_mask, size=image.shape[:2], mode="bilinear", align_corners=False
) #resize output mask to match original image size
mask = mask[0].argmax(0)  # Remains as a PyTorch tensor on the GPU
# #choose class with highest probability for each pixel
#then moves resulting tensor to CPU and converts to np array for visualization

In [8]:
from tqdm import tqdm
from PIL import Image, UnidentifiedImageError

In [9]:
from PIL import Image, ExifTags
import os

def extract_gps_coordinates(image_path):
    """Extract latitude and longitude from an image's EXIF GPS data."""
    try:
        image = Image.open(image_path)
        exif_data = image._getexif()
        if not exif_data:
            return None

        # Convert EXIF tag IDs to human-readable names
        exif = {ExifTags.TAGS.get(tag, tag): value for tag, value in exif_data.items()}

        if "GPSInfo" not in exif:
            return None

        gps_info = exif["GPSInfo"]

        # Extract GPS latitude and longitude
        latitude = gps_info.get(2)  # GPSLatitude
        latitude_ref = gps_info.get(1)  # 'N' or 'S'
        longitude = gps_info.get(4)  # GPSLongitude
        longitude_ref = gps_info.get(3)  # 'E' or 'W'

        if latitude and longitude and latitude_ref and longitude_ref:
            # Convert (degrees, minutes, seconds) to decimal format
            lat = latitude[0] + latitude[1] / 60.0 + latitude[2] / 3600.0
            lon = longitude[0] + longitude[1] / 60.0 + longitude[2] / 3600.0

            # Adjust for N/S and E/W
            if latitude_ref == 'S':
                lat = -lat
            if longitude_ref == 'W':
                lon = -lon

            return lat, lon

        return None
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None

# Path to images in your cloned GitHub repository
github_folder = "/Users/leeelyehezkel/PycharmProjects/PythonProject/Turkey-Building-Data/Building images"

# Array to store filename and GPS coordinates
image_gps_data = []

# Iterate through images and extract GPS coordinates
for filename in os.listdir(github_folder):
    if filename.lower().endswith(('.jpg', '.jpeg')):  # Only check JPG images
        image_path = os.path.join(github_folder, filename)
        gps_coordinates = extract_gps_coordinates(image_path)
        if gps_coordinates:
            image_gps_data.append({"filename": filename, "latitude": gps_coordinates[0], "longitude": gps_coordinates[1]})

# Print results
for entry in image_gps_data:
    print(f"File: {entry['filename']}, Latitude: {entry['latitude']}, Longitude: {entry['longitude']}")

# Optional: Convert to a DataFrame (if needed for analysis)
import pandas as pd
df = pd.DataFrame(image_gps_data)
print(df)

File: {A0D18F36-EA14-4F20-8765-A379DCEAA9BB}.jpg, Latitude: 36.23093380555556, Longitude: 36.200752083333334
File: {06261D46-239D-4BDA-8AEB-F761F35C0249}.jpg, Latitude: 37.75729919444444, Longitude: 38.25585708333333
File: {4A5FE300-7542-4021-A196-C09A0E8D542E}.jpg, Latitude: 37.765188055555555, Longitude: 38.26533038888889
File: {F8E6D8C1-7DC7-40D2-9BE2-A7D5F653E380}.jpg, Latitude: 36.259728611111115, Longitude: 36.20639278888889
File: {C79931AC-1A1F-4906-8761-5E276EC978EF}.jpg, Latitude: 36.234574880555556, Longitude: 36.16567141666666
File: {2A8304E5-B65C-4A49-A745-FD29854CEC7A}.jpg, Latitude: 37.7519047, Longitude: 38.24508180555556
File: {C165CBE8-C191-4C8A-AD79-ACB5DC61936C}.jpg, Latitude: 37.58274658333334, Longitude: 36.882894666666665
File: {64168C52-3BE4-43EA-8E3B-0999F05147AC}.jpg, Latitude: 36.20614769444445, Longitude: 36.156423877777776
File: {246A5A81-F042-4258-A6C2-A4F061E2D0BB}.jpg, Latitude: 36.22787969444445, Longitude: 36.14938769444444
File: {3F02D8E9-773E-47CD-894

In [17]:
import pandas as pd

# Define the file path for the Excel file
output_file = '/Users/leeelyehezkel/PycharmProjects/PythonProject/Turkey-Building-Data/gps_data.xlsx'

# Save the DataFrame to an Excel file
df.to_excel(output_file, index=False)

print(f"DataFrame saved as Excel file at: {output_file}")

DataFrame saved as Excel file at: /Users/leeelyehezkel/PycharmProjects/PythonProject/Turkey-Building-Data/gps_data.xlsx


In [18]:
import os
import subprocess

# Define the repository path
repo_path = '/Users/leeelyehezkel/PycharmProjects/PythonProject/Turkey-Building-Data'

# Change to the repository directory
os.chdir(repo_path)

# Stage the updated Excel file
subprocess.run(['git', 'add', 'gps_data.xlsx'], check=True)

# Commit the changes
subprocess.run(['git', 'commit', '-m', 'Update GPS data Excel file'], check=True)

# Push the changes to the remote repository
subprocess.run(['git', 'push'], check=True)

print("Changes have been pushed to the remote GitHub repository.")

[main 75abf4e] Update GPS data Excel file
 Committer: LeeEl Yehezkel <leeelyehezkel@LeeEls-MacBook-Air.local>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly:

    git config --global user.name "Your Name"
    git config --global user.email you@example.com

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 1 file changed, 0 insertions(+), 0 deletions(-)
Changes have been pushed to the remote GitHub repository.


To https://github.com/leeelyehezkel/Turkey-Building-Data.git
   f1846ea..75abf4e  main -> main


Load pretrained model and train on new photos with new clases:
Modifying the Model Architecture:

Load the Pretrained Model:

Start with the SegFormer model pretrained on Cityscapes (which outputs 19 classes).

Created a New Segmentation Head:

To predict 9 classes (all 5 triage categories plus people, background, and machinery), replace the final layer (the segmentation head) with a new convolutional layer that outputs 9 channels.

Copied Pretrained Weights:

For the first 19 channels (corresponding to the Cityscapes classes), copy over the pretrained weights. The new 9 channels (for the new classes) are initialized randomly. This helps preserve the learned features for the original classes while allowing the network to learn the new classes.

In [19]:
# Load the pretrained model (Cityscapes – 19 classes)
checkpoint = "smp-hub/segformer-b2-1024x1024-city-160k"
model = smp.from_pretrained(checkpoint).to(device)

# Define number of classes:
total_classes = 9

# Replace the segmentation head with a new one that outputs 9 channels.
# (Here we assume the segmentation head is a simple Conv2d layer.)
old_head = model.segmentation_head  # typically a SegmentationHead layer
# Get in_channels from the first layer within the SegmentationHead
in_channels = old_head[0].in_channels  # Accessing the in_channels of the Conv2d layer

# Create new head
new_head = torch.nn.Conv2d(in_channels, total_classes, kernel_size=1)

# Optionally, copy the pretrained weights for the first 19 classes
# with torch.no_grad():
    # new_head.weight[:num_cityscapes_classes] = old_head[0].weight
    # new_head.bias[:num_cityscapes_classes] = old_head[0].bias

# The above lines refer to the pretrained weights that take the info from all of
# the previous layers and give the labels names as the original 19 classes.  Therefore, irrelevant

# Replace the head in the model
model.segmentation_head = new_head.to(device)

Preparing the Combined Dataset

Created a Custom Dataset Class:

Define a PyTorch Dataset class (BuildingDataset) that reads images and their corresponding mask files from designated folders.

Preprocessing and Transformations:

Apply the same preprocessing pipeline (using Albumentations) as used during training of the pretrained model. This ensures that the input images are normalized and resized in a consistent way.

Pairing Images with Masks:

The dataset class includes logic to match each mask with its corresponding image using filename patterns.

In [21]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

import os
import numpy as np
from PIL import Image
import albumentations as A
import torch
from torch.utils.data import Dataset

# -------------------------
# 1. Define the Dataset with Extra Augmentation for New Data
# -------------------------
class BuildingDataset(Dataset):
    def __init__(self, image_folder, mask_folder, preprocessing, base_transforms=None, augment=False, image_size=(1024, 1024)):
        """
        image_folder: Path to building images.
        mask_folder: Path to corresponding mask images.
        preprocessing: Preprocessing pipeline from the pretrained checkpoint.
        base_transforms: Any basic transforms (applied to all data).
        augment: If True, extra augmentation will be applied.
        """
        self.image_folder = image_folder
        self.mask_folder = mask_folder
        self.preprocessing = preprocessing
        self.base_transforms = base_transforms
        self.augment = augment
        self.pairs = self._collect_pairs()
        self.image_size = image_size # Store image_size

        # Define extra augmentation pipeline if augment flag is True
        if self.augment:
            self.augmentation_pipeline = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=45, p=0.5),
                A.ColorJitter(p=0.5),
                # Add more augmentations as needed
            ])
        else:
            self.augmentation_pipeline = None

    def _collect_pairs(self):
        pairs = []
        mask_files = [f for f in os.listdir(self.mask_folder) if f.startswith("Label_") and f.endswith(".png")]
        building_files = os.listdir(self.image_folder)

        def find_building_image(base_img_name, building_files):
            for file in building_files:
                if file.startswith(base_img_name):
                    return file
            return None

        for mask_file in mask_files:
            parts = mask_file.split('_')
            if len(parts) < 3:
                continue
            image_part = "_".join(parts[2:])
            base_img_name = os.path.splitext(image_part)[0]
            building_img_file = find_building_image(base_img_name, building_files)
            if building_img_file:
                original_path = os.path.join(self.image_folder, building_img_file)
                mask_path = os.path.join(self.mask_folder, mask_file)
                pairs.append((original_path, mask_path))
        return pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        image_path, mask_path = self.pairs[idx]
        image = np.array(Image.open(image_path).convert("RGB"))
        mask = np.array(Image.open(mask_path))

        # Resize the image and mask to a consistent size
        resize_transform = A.Resize(height=self.image_size[0], width=self.image_size[1]) # Resize using Albumentations
        resized = resize_transform(image=image, mask=mask)
        image, mask = resized["image"], resized["mask"]

        # Apply base transforms if provided
        if self.base_transforms:
            augmented = self.base_transforms(image=image, mask=mask)
            image, mask = augmented["image"], augmented["mask"]

        # Apply extra augmentation only for the new dataset if enabled
        if self.augment and self.augmentation_pipeline:
            augmented = self.augmentation_pipeline(image=image, mask=mask)
            image, mask = augmented["image"], augmented["mask"]

        # Preprocess image (e.g., normalization) using the pretrained pipeline
        preprocessed = self.preprocessing(image=image)
        image = preprocessed["image"]
        image = torch.as_tensor(image).permute(2, 0, 1).float()
        mask = torch.as_tensor(mask).long()

        return image, mask

# Example usage for the new dataset with extr augmentation:
image_folder = "/Users/leeelyehezkel/PycharmProjects/PythonProject/Turkey-Building-Data/Building images"
mask_folder = '/Users/leeelyehezkel/PycharmProjects/PythonProject/Turkey-Building-Data/PixelLabelData2'
# 'preprocessing' should be defined as per your pretrained model's requirements
building_dataset = BuildingDataset(image_folder, mask_folder, preprocessing, base_transforms=None, augment=True)


# -------------------------
# 2. Freeze the Encoder Layers of the Model
# -------------------------
# Assume your model is loaded as shown:
# model = smp.from_pretrained(checkpoint).eval().to(device)

# Freeze all parameters in the encoder to prevent them from updating during training.
for param in model.encoder.parameters():
    param.requires_grad = False

print("Encoder layers frozen. Only the head and later layers will be trained.")

Encoder layers frozen. Only the head and later layers will be trained.


In [23]:
import os

# Get a list of all building image filenames
all_building_files = os.listdir(image_folder)

# Get a list of mask filenames (they start with "Label_")
mask_files = [f for f in os.listdir(mask_folder) if f.startswith("Label_") and f.endswith(".png")]

def extract_base_name(mask_filename):
    # Assumes mask filename like "Label_2_BE123.png", returns "BE123"
    parts = mask_filename.split('_')
    if len(parts) < 3:
        return None
    image_part = "_".join(parts[2:])
    base_img_name = os.path.splitext(image_part)[0]
    return base_img_name

# Determine the set of labeled building image base names
labeled_bases = set()
for mask_file in mask_files:
    base = extract_base_name(mask_file)
    if base:
        labeled_bases.add(base)

# Identify unlabeled images by checking if the filename's base is not in labeled_bases
unlabeled_files = []
for file in all_building_files:
    base_name = os.path.splitext(file)[0]
    if base_name not in labeled_bases:
        unlabeled_files.append(os.path.join(image_folder, file))

print(f"Found {len(unlabeled_files)} unlabeled building images.")

# Create val_pairs for labeled images: a pair of (image_path, mask_path)
val_pairs = []
for mask_file in mask_files:
    base = extract_base_name(mask_file)
    if base:
        # Find the corresponding image whose filename starts with the base name
        for file in all_building_files:
            if file.startswith(base):
                image_path = os.path.join(image_folder, file)
                mask_path = os.path.join(mask_folder, mask_file)
                val_pairs.append((image_path, mask_path))
                break

print(f"Found {len(val_pairs)} labeled (validation) pairs.")


Found 38 unlabeled building images.
Found 311 labeled (validation) pairs.


Setting Up the Training Pipeline

DataLoader:

Create a DataLoader to iterate over our dataset in batches, shuffling the data for training.

Loss Function:

Use the CrossEntropyLoss, which is suitable for multi-class segmentation problems (each pixel belongs to one class).

Optimizer:

Use the Adam optimizer with a specified learning rate to update the model's weights during training.

Training Loop:

The loop iterates over several epochs, processes batches of images and masks, performs forward passes through the model, computes the loss, and backpropagates the error to update weights.

In [24]:
import os
import numpy as np
from PIL import Image

# Get a list of all mask files (assuming PNG format)
mask_files = [os.path.join(mask_folder, f) for f in os.listdir(mask_folder) if f.endswith('.png')]

# Define number of classes (adjust if necessary; here we assume classes 0 through 8)
num_classes = 9

# Initialize an array to accumulate pixel counts for each class
class_pixel_counts = np.zeros(num_classes, dtype=np.int64)
total_pixels = 0

# Loop through all mask images
for mask_file in mask_files:
    try:
        mask_img = np.array(Image.open(mask_file))
    except Exception as e:
        print(f"Error reading {mask_file}: {e}")
        continue

    # Flatten the mask to 1D array and count pixels per class
    counts = np.bincount(mask_img.flatten(), minlength=num_classes)
    class_pixel_counts += counts
    total_pixels += mask_img.size

# Compute the percentage of pixels for each class
percentages = (class_pixel_counts / total_pixels) * 100

print("Pixel percentages per class:")
for i, perc in enumerate(percentages):
    print(f"Class {i}: {perc:.2f}%")

Pixel percentages per class:
Class 0: 47.18%
Class 1: 6.95%
Class 2: 2.86%
Class 3: 2.23%
Class 4: 13.61%
Class 5: 4.94%
Class 6: 3.25%
Class 7: 1.84%
Class 8: 17.13%


In [27]:
import torch
from torch.utils.data import random_split, DataLoader
import torch.optim as optim

# Assuming building_dataset is already defined from your earlier code.
dataset_size = len(building_dataset)
val_size = int(0.2 * dataset_size)       # e.g., 20% for validation
train_size = dataset_size - val_size

# Split the dataset into training and validation sets
train_dataset, val_dataset = random_split(building_dataset, [train_size, val_size])

# Create DataLoaders for each split
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)

# Suppose you computed class_pixel_counts via bincount
total_pixels = sum(class_pixel_counts)
class_weights = [total_pixels / count for count in class_pixel_counts]
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

# Define your optimizer (Adam) with an initial learning rate.
optimizer = optim.Adam(model.parameters(), lr=1e-3)
# Define a scheduler that reduces the LR if the validation loss doesn't improve.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

epsilon = 0.005  # Minimum relative improvement (0.5% improvement)
patience_threshold = 3  # Stop if improvement is below epsilon for 3 consecutive epochs
patience_counter = 0
previous_val_loss = None
train_losses = []
val_losses = []

num_epochs = 50  # Set maximum epochs

for epoch in range(num_epochs):
    model.train()  # Training mode
    train_loss = 0.0
    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)

        # Forward pass
        outputs = model(images)
        outputs = torch.nn.functional.interpolate(
            outputs, size=masks.shape[1:], mode="bilinear", align_corners=False
        )

        loss = criterion(outputs, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # Validation phase
    model.eval()  # Evaluation mode
    val_loss = 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            outputs = torch.nn.functional.interpolate(
                outputs, size=masks.shape[1:], mode="bilinear", align_corners=False
            )
            loss = criterion(outputs, masks)
            val_loss += loss.item()
    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    # Step the scheduler with the validation loss
    scheduler.step(avg_val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, LR: {current_lr:.6f}")

    # Early stopping: check absolute difference in validation loss
    if previous_val_loss is not None:
        loss_diff = abs(avg_val_loss - previous_val_loss)
        if loss_diff < epsilon:
            patience_counter += 1
            print(f"Loss diff {loss_diff:.5f} below epsilon. Patience: {patience_counter}/{patience_threshold}")
        else:
            patience_counter = 0
    previous_val_loss = avg_val_loss

    if patience_counter >= patience_threshold:
        print("Early stopping triggered!")
        break
plt.figure(figsize=(10, 6))
epochs = range(1, len(train_losses) + 1)
plt.plot(epochs, train_losses, 'r-', label='Train Loss')
plt.plot(epochs, val_losses, 'b-', label='Val Loss')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Train and Val Loss Over Epochs')
plt.legend()

# Automatically adjust the x and y limits based on data
plt.xlim(1, num_epochs)
plt.ylim(min(min(train_losses), min(val_losses)) * 0.95, max(max(train_losses), max(val_losses)) * 1.05)


import matplotlib.pyplot as plt

# Plot the training and validation losses
plt.figure(figsize=(10, 6))
epochs = range(1, len(train_losses) + 1)
plt.plot(epochs, train_losses, 'r-', label='Train Loss')
plt.plot(epochs, val_losses, 'b-', label='Val Loss')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Train and Val Loss Over Epochs')
plt.legend()

# Automatically adjust the x and y limits based on data
plt.xlim(1, len(train_losses))
plt.ylim(min(min(train_losses), min(val_losses)) * 0.95, max(max(train_losses), max(val_losses)) * 1.05)

# Display the plot
plt.show()

/Users/leeelyehezkel/PycharmProjects/SegFormer/.venv/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<string>", line 1, in <module>
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/Library/Framework

RuntimeError: DataLoader worker (pid(s) 6020, 6021) exited unexpectedly

In [28]:
import torch
from torch.utils.data import random_split, DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt

# Assuming building_dataset is already defined from your earlier code.
dataset_size = len(building_dataset)
val_size = int(0.2 * dataset_size)  # e.g., 20% for validation
train_size = dataset_size - val_size

# Split the dataset into training and validation sets
train_dataset, val_dataset = random_split(building_dataset, [train_size, val_size])

# Create DataLoaders for each split
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)

# Suppose you computed class_pixel_counts via bincount
total_pixels = sum(class_pixel_counts)
class_weights = [
    total_pixels / count if count > 0 else 0 for count in class_pixel_counts
]  # Avoid division by zero
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

# Define your optimizer (Adam) with an initial learning rate.
optimizer = optim.Adam(model.parameters(), lr=1e-3)
# Define a scheduler that reduces the LR if the validation loss doesn't improve.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2, verbose=True)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

epsilon = 0.005  # Minimum relative improvement (0.5% improvement)
patience_threshold = 3  # Stop if improvement is below epsilon for 3 consecutive epochs
patience_counter = 0
previous_val_loss = None
train_losses = []
val_losses = []

num_epochs = 50  # Set maximum epochs

for epoch in range(num_epochs):
    model.train()  # Training mode
    train_loss = 0.0
    for images, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)

        # Forward pass
        outputs = model(images)
        outputs = torch.nn.functional.interpolate(
            outputs, size=masks.shape[1:], mode="bilinear", align_corners=False
        )

        loss = criterion(outputs, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # Validation phase
    model.eval()  # Evaluation mode
    val_loss = 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            outputs = torch.nn.functional.interpolate(
                outputs, size=masks.shape[1:], mode="bilinear", align_corners=False
            )
            loss = criterion(outputs, masks)
            val_loss += loss.item()
    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    # Step the scheduler with the validation loss
    scheduler.step(avg_val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, LR: {current_lr:.6f}")

    # Early stopping: check absolute difference in validation loss
    if previous_val_loss is not None:
        loss_diff = abs(avg_val_loss - previous_val_loss)
        if loss_diff < epsilon:
            patience_counter += 1
            print(f"Loss diff {loss_diff:.5f} below epsilon. Patience: {patience_counter}/{patience_threshold}")
        else:
            patience_counter = 0
    previous_val_loss = avg_val_loss

    if patience_counter >= patience_threshold:
        print("Early stopping triggered!")
        break

# Plot the training and validation losses
plt.figure(figsize=(10, 6))
epochs = range(1, len(train_losses) + 1)
plt.plot(epochs, train_losses, 'r-', label='Train Loss')
plt.plot(epochs, val_losses, 'b-', label='Val Loss')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Train and Val Loss Over Epochs')
plt.legend()

# Automatically adjust the x and y limits based on data
plt.xlim(1, len(train_losses))
plt.ylim(min(min(train_losses), min(val_losses)) * 0.95, max(max(train_losses), max(val_losses)) * 1.05)

# Display the plot
plt.show()

/Users/leeelyehezkel/PycharmProjects/SegFormer/.venv/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<string>", line 1, in <module>
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/Library/Framework

RuntimeError: DataLoader worker (pid(s) 6185) exited unexpectedly

In [31]:
# Save BuildingDataset in a separate file, e.g., dataset.py
from BuildingDataset import BuildingDataset  # Import the dataset class
import torch
from torch.utils.data import random_split, DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt

if __name__ == "__main__":
    # Initialize the dataset
    image_folder = "/Users/leeelyehezkel/PycharmProjects/PythonProject/Turkey-Building-Data/Building images"
    mask_folder = "/Users/leeelyehezkel/PycharmProjects/PythonProject/Turkey-Building-Data/PixelLabelData2"
    building_dataset = BuildingDataset(image_folder, mask_folder, preprocessing, augment=True)

    # Split the dataset
    dataset_size = len(building_dataset)
    val_size = int(0.2 * dataset_size)
    train_size = dataset_size - val_size
    train_dataset, val_dataset = random_split(building_dataset, [train_size, val_size])

    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)

    # Compute class weights
    total_pixels = sum(class_pixel_counts)
    class_weights = [
        total_pixels / count if count > 0 else 0 for count in class_pixel_counts
    ]
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

    # Define optimizer, scheduler, and loss function
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
    criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

    # Training loop
    for epoch in range(50):
        model.train()
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(device), masks.to(device)
                outputs = model(images)
                val_loss += criterion(outputs, masks).item()
        scheduler.step(val_loss)
        print(f"Epoch {epoch+1}, Val Loss: {val_loss:.4f}")

/Users/leeelyehezkel/PycharmProjects/SegFormer/.venv/lib/python3.9/site-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.6 (you have 1.4.21). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
  File "/Users/leeelyehezkel/PycharmProjects/SegFormer/.venv/lib/python3.9/site-packages/numpy/random/_pickle.py", line 34, in __bit_generator_ctor
    raise ValueError(str(bit_generator_name) + ' is not a known '
ValueError: <class 'num

RuntimeError: DataLoader worker (pid(s) 6845) exited unexpectedly

In [1]:
import os
print(os.getcwd())

/Users/leeelyehezkel/PycharmProjects/PythonProject/Turkey-Building-Data
